# Forensic LLM Graph

Use a **T4 GPU** runtime. For the full pipeline, upload `documents.json` to `/content/`, set `reuse_saved_ner = False` and enable the `HF_TOKEN` Colab Secret for Minerva. To rebuild only the expanded graph, upload your saved hybrid NER file to `/content/`, set `reuse_saved_ner = True` and enter its name in `saved_ner_name`.

Choose a new two-digit run number, select **Runtime → Run all**, and download the numbered ZIP from the Files panel. In graph-only mode the ZIP contains the relation plan, Qwen responses and graph. The uploaded NER remains the source for inspecting graph evidence.

If the runtime still holds its files after an interruption, rerun with the same options. Completed stages are reused, and the NER and relation reviews resume from their checkpoints.


## 1. Set up Colab


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import userdata

run_number = "01"  # @param {type:"string"}
reuse_saved_ner = False  # @param {type:"boolean"}
saved_ner_name = "01_ner_corpus.json"  # @param {type:"string"}
relation_budget = 100  # @param {type:"integer"}
initial_review_check = 5  # @param {type:"integer"}
if len(run_number) != 2 or not run_number.isascii() or not run_number.isdecimal():
    raise ValueError("Use two digits for the run number, for example 01.")

workspace = Path("/content")
uploaded_corpus = workspace / "documents.json"
if Path(saved_ner_name).name != saved_ner_name or not saved_ner_name.endswith(".json"):
    raise ValueError("Use the filename of the saved NER JSON.")
if relation_budget < 1:
    raise ValueError("The relation budget must be positive.")
if initial_review_check < 0:
    raise ValueError("Use a non-negative number of passages for the initial review check.")
uploaded_ner = workspace / saved_ner_name
if reuse_saved_ner and not uploaded_ner.is_file():
    raise FileNotFoundError(f"Upload {saved_ner_name} to /content/ first.")
if not reuse_saved_ner and not uploaded_corpus.is_file():
    raise FileNotFoundError("Upload the prepared documents.json to /content/ first.")

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a T4 GPU runtime in Colab.")
subprocess.run(["nvidia-smi", "-L"], check=True)

try:
    hf_token = (userdata.get("HF_TOKEN") or "").strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    hf_token = ""
if not reuse_saved_ner and not hf_token:
    raise RuntimeError("Enable the HF_TOKEN secret for Minerva.")
if hf_token and (not hf_token.isascii() or not hf_token.isprintable() or " " in hf_token):
    raise RuntimeError("Paste the token alone into HF_TOKEN, on one line.")

repo = workspace / "forensic-llm-graph"
output_dir = repo / "outputs"
baseline_path = output_dir / f"{run_number}_ner_baseline.json"
ner_path = uploaded_ner if reuse_saved_ner else output_dir / f"{run_number}_ner_corpus.json"
plan_path = output_dir / f"{run_number}_relation_plan.json"
reviews_path = output_dir / f"{run_number}_relation_reviews.json"
graph_path = output_dir / f"{run_number}_relation_graph.json"

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)],
        check=True,
    )
elif not plan_path.exists() and (reuse_saved_ner or not baseline_path.exists()):
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)

revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(f"Pipeline revision: {revision}", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=repo, check=True,
)

if not reuse_saved_ner:
    from huggingface_hub import hf_hub_download

    hybrid_config = json.loads((repo / "configs/hybrid.json").read_text(encoding="utf-8"))
    minerva = hybrid_config["perplexity"]
    hf_hub_download(minerva["model"], "config.json", revision=minerva["revision"], token=hf_token)
model_env = {**os.environ}
if hf_token:
    model_env["HF_TOKEN"] = hf_token
del hf_token


def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def run_stage(path, stage, *args, environment=None, **expected):
    """Run a pipeline stage or verify the provenance of its saved output."""
    if path.is_file():
        print(f"Reusing {path.name}.")
    else:
        command = [sys.executable, "-u", *map(str, args), "--output", str(path)]
        with subprocess.Popen(
            command, cwd=repo, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        ) as process:
            try:
                for line in process.stdout:
                    if line.startswith("Perplexity:"):
                        done, total = map(int, line.split(":", 1)[1].split("/"))
                        if total > 100 and done % 100 and done != total:
                            continue
                    print(line, end="", flush=True)
                return_code = process.wait()
            except KeyboardInterrupt:
                process.terminate()
                process.wait()
                raise
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)

    with path.open(encoding="utf-8") as handle:
        metadata = json.load(handle)["run"]
    if metadata.get("stage") != stage or any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError(f"{path.name} does not match this run. Use a new run number.")
    return metadata


## 2. Extract entities with rules and spaCy

In full mode, the script processes the full corpus on CPU and saves document text and candidate spans.


In [ ]:
if not reuse_saved_ner:
    corpus_path = repo / "data/processed/documents.json"
    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(uploaded_corpus, corpus_path)

    baseline = run_stage(
        baseline_path, "first_pass", "scripts/run_pipeline.py",
        input_sha256=sha256(corpus_path), mode="rules_and_spacy",
        git={"commit": revision, "uncommitted_changes": False},
    )
    document_count = baseline["document_count"]
    print(f"Documents: {document_count:,}. First-pass candidates: {baseline['candidate_count']:,}.")


## 3. Score passages and review entities

In full mode, Minerva scores all passages and Qwen reviews the selected passages under the configured budget. The script saves checkpoints as it runs.


In [ ]:
if not reuse_saved_ner:
    checkpoint = ner_path.with_suffix(".checkpoints.jsonl")
    resume = ["--resume"] if checkpoint.is_file() else []
    ner = run_stage(
        ner_path, "hybrid", "scripts/run_hybrid.py", "--input", baseline_path, *resume,
        environment=model_env, input_sha256=sha256(baseline_path), scope="extended",
        document_count=document_count, git=baseline["git"],
    )
    print(f"Entity review statuses: {ner['review_status_counts']}.")


## 4. Review relationships and build the expanded graph

The plan selects passages for Qwen using the configured budget. The CPU builder adds contextual facts and verified links across documents and retains the offsets or source metadata supporting each edge.


In [ ]:
if reuse_saved_ner:
    with ner_path.open(encoding="utf-8-sig") as handle:
        saved = json.load(handle)
    if saved.get("run", {}).get("stage") != "hybrid" or not saved.get("documents"):
        raise ValueError("Upload a saved hybrid NER JSON, for example 01_ner_corpus.json.")
    document_count = len(saved["documents"])
    del saved
ner_sha256 = sha256(ner_path)
plan = run_stage(
    plan_path, "relation_review_plan", "scripts/graph.py", "prepare",
    "--input", ner_path, "--context", "--max-passages", relation_budget,
    input_sha256=ner_sha256, scope="corpus", profile="context", max_passages=relation_budget, review_mode="direct_proposals",
)
reviews = run_stage(
    reviews_path, "relation_review", "scripts/graph.py", "review",
    "--plan", plan_path, "--check-first", initial_review_check, environment=model_env, input_sha256=ner_sha256,
    plan_sha256=sha256(plan_path), selected_passages=plan["selected_passages"], profile="context",
)
print(f"Relation review statuses: {reviews['statuses']}.")

graph = run_stage(
    graph_path, "relation_graph", "scripts/graph.py", "build",
    "--input", ner_path, "--context", "--relation-reviews", reviews_path,
    input_sha256=ner_sha256, relation_review_sha256=sha256(reviews_path), profile="context",
    link_config_sha256=sha256(repo / "configs/graph_links.json"),
)
if graph["nodes_by_type"].get("DOCUMENT", 0) != document_count:
    raise ValueError("The graph does not cover all documents in the input.")
print(f"Nodes: {graph['nodes_by_type']}")
print(f"Edges: {graph['edges_by_type']}")


## 5. Download the outputs

Download the ZIP from the Colab Files panel and extract it into the local `outputs/` folder. Keep the original NER JSON for evidence inspection.


In [ ]:
bundle_path = workspace / f"{run_number}_run.zip"
temporary_bundle = bundle_path.with_suffix(".zip.tmp")
with ZipFile(temporary_bundle, "w", compression=ZIP_DEFLATED, compresslevel=1) as bundle:
    artifacts = (plan_path, reviews_path, graph_path)
    if not reuse_saved_ner:
        artifacts = (baseline_path, ner_path, *artifacts)
    for path in artifacts:
        bundle.write(path, arcname=path.name)
temporary_bundle.replace(bundle_path)
print(f"Download {bundle_path} from the Colab Files panel.")
